Using Selected Features, Predict Genre

Using Macro F1 as primary metric, along with weighted F1 and Top 3 accuracy 

Also include macro roc auc , macro pr auc, and log loss

In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, LabelBinarizer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    f1_score,
    average_precision_score,
    roc_auc_score,
    top_k_accuracy_score,
    log_loss
)

from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [3]:
def evaluate_model(y_true, y_prob):
    total_classes = y_prob.shape[1]

    # Derive class predictions from highest probability
    y_pred = y_prob.argmax(axis=1)
    
    # Binarize true labels for multiclass PR-AUC calculations
    lb = LabelBinarizer()
    lb.fit(range(total_classes))  # Ensure all classes are considered
    y_true_binarized = lb.transform(y_true)
    
    # Calculate metrics
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    # Identify classes that contain BOTH positive and negative examples in y_true
    # This prevents Scikit-Learn from crashing if a rare class lacks variance
    valid_auc_classes = (y_true_binarized.sum(axis=0) > 0) & (y_true_binarized.sum(axis=0) < len(y_true))
    
    if np.all(valid_auc_classes):
        macro_prauc = average_precision_score(y_true_binarized, y_prob, average='macro')
        macro_rocauc = roc_auc_score(y_true_binarized, y_prob, multi_class='ovr', average='macro')
    else:
        # Calculate macro metrics only over classes that have valid target distributions
        macro_prauc = average_precision_score(y_true_binarized[:, valid_auc_classes], y_prob[:, valid_auc_classes], average='macro')
        macro_rocauc = roc_auc_score(y_true_binarized[:, valid_auc_classes], y_prob[:, valid_auc_classes], multi_class='ovr', average='macro')
    
    
    top_3_acc = top_k_accuracy_score(y_true, y_prob, k=3, labels=range(total_classes))
    logloss = log_loss(y_true, y_prob, labels=range(total_classes))
    
    return {
        "Macro F1": macro_f1,
        "Weighted F1": weighted_f1,
        "Macro PR-AUC": macro_prauc,
        "Macro ROC AUC": macro_rocauc,
        "Top-3 Accuracy": top_3_acc,
        "Log-Loss": logloss
    }

Read in the datasets

In [2]:
X_train = pd.read_csv('data/scaled_X_train.csv')
X_test = pd.read_csv('data/scaled_X_test.csv')
y_train = pd.read_csv('data/y_train.csv')
y_test = pd.read_csv('data/y_test.csv')

Selected Features

In [3]:
selected_features = [
    'bpm',
    'danceability',
    'onset_rate',
    'average_loudness',
    'dynamic_complexity',
    'mfcc_zero_mean',
    'tuning_frequency',
    'tuning_equal_tempered_deviation',
    'mood_happy_prob',
    'mood_aggressive_prob',
    'mood_acoustic',
    'mood_electronic',
    'timbre',
    'voice_instrumental'
]

Keep only selected features

In [4]:
X_train = X_train[selected_features]
X_test = X_test[selected_features]

In [5]:
X_train.columns

Index(['bpm', 'danceability', 'onset_rate', 'average_loudness',
       'dynamic_complexity', 'mfcc_zero_mean', 'tuning_frequency',
       'tuning_equal_tempered_deviation', 'mood_happy_prob',
       'mood_aggressive_prob', 'mood_acoustic', 'mood_electronic', 'timbre',
       'voice_instrumental'],
      dtype='str')

In [6]:
X_train.shape

(2860386, 14)

In [7]:
X_test.shape

(715097, 14)

In [8]:
y_train['main_genre'].value_counts()

main_genre
2    769102
9    685703
7    276846
5    244855
0    239686
6    212991
4    181843
8     95952
3     83436
1     69972
Name: count, dtype: int64

In [ ]:
# Electronic     961377 = 2
# Rock           857129 = 9
# Pop            346058 = 7
# Jazz/Blues     306069 = 5
# Classical      299607 = 0
# Metal          266239 = 6
# Hip-Hop/R&B    227304 = 4
# Punk           119940 = 8
# Folk           104295 = 3
# Country         87465 = 1

In [12]:
import pickle
import os

# Compute sample weights dynamically
sample_weights_train = compute_sample_weight(class_weight='balanced', y=y_train)

# All models configured to address the class imbalance
models = {
    "Random Forest": RandomForestClassifier(
        class_weight='balanced_subsample', 
        n_jobs=-1, 
        random_state=42
    ),
    
    "LightGBM": lgb.LGBMClassifier(
        objective='multiclass',
        num_class=len(set(y_train)) ,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ),
    
    "XGBoost": XGBClassifier(
        objective='multi:softprob', 
        num_class=len(set(y_train)) , 
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    ),
    
    "CatBoost": CatBoostClassifier(
        loss_function='MultiClass', 
        auto_class_weights='Balanced',
        random_state=42,
        verbose=50  
    )
}

performance_results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")

    # separate out each model so can view results later
    # added current_model variable to track current model for pickle saving

    if name == "XGBoost":
        xgb_model = model
        xgb_model.fit(X_train, y_train, sample_weight=sample_weights_train)
        current_model = xgb_model

    elif name == "LightGBM":
        lgb_model = model
        lgb_model.fit(
            X_train, y_train,
            sample_weight=sample_weights_train, 
            eval_set=[(X_test, y_test)],
            eval_metric="multi_logloss",
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)] 
        )
        current_model = lgb_model

    elif name == "CatBoost":
        catboost_model = model
        catboost_model.fit(X_train, y_train)
        current_model = catboost_model

    else:
        rf_model = model
        rf_model.fit(X_train, y_train)
        current_model = rf_model

    # Save Model to Pickle Files
    # Cleans up spaces in name for the filename (i.e., "Random Forest" -> "random_forest_model.pkl")
    clean_name = name.lower().replace(' ', '_')
    filepath = os.path.join("models", f"{clean_name}_genre_model_features_selected.pkl")

    print(f"Saving {name} to {filepath}...")
    with open(filepath, "wb") as file:
        pickle.dump(current_model, file)


    print(f"Evaluating {name} on validation data...")
    if name == "CatBoost":
        y_prob_test = catboost_model.predict_proba(X_test)
    elif name == "LightGBM":
        y_prob_test = lgb_model.predict_proba(X_test)
    elif name == "XGBoost":
        y_prob_test = xgb_model.predict_proba(X_test)
    else:
        y_prob_test = rf_model.predict_proba(X_test)

    # Execute the evaluation function
    metrics = evaluate_model(y_test, y_prob_test)
    performance_results[name] = metrics

print("\n" + "="*50)
print("FINAL MODEL COMPARISON RESULTS")
print("="*50)

df_results = pd.DataFrame(performance_results).T
print(df_results.to_string(formatters={c: '{:,.4f}'.format for c in df_results.columns}))


Training Random Forest...


c:\Users\627700\Documents\Music-Analytics-Prediction-RAG\.venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Saving Random Forest to models\random_forest_genre_model_features_selected.pkl...
Evaluating Random Forest on validation data...

Training LightGBM...


c:\Users\627700\Documents\Music-Analytics-Prediction-RAG\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:103: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\627700\Documents\Music-Analytics-Prediction-RAG\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
c:\Users\627700\Documents\Music-Analytics-Prediction-RAG\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
c:\Users\627700\Documents

Saving LightGBM to models\lightgbm_genre_model_features_selected.pkl...
Evaluating LightGBM on validation data...

Training XGBoost...
Saving XGBoost to models\xgboost_genre_model_features_selected.pkl...
Evaluating XGBoost on validation data...

Training CatBoost...
Learning rate set to 0.118741
0:	learn: 2.1717246	total: 1.26s	remaining: 20m 57s
50:	learn: 1.6401956	total: 37.6s	remaining: 11m 39s
100:	learn: 1.6088803	total: 1m 14s	remaining: 11m 5s
150:	learn: 1.5940263	total: 1m 52s	remaining: 10m 30s
200:	learn: 1.5831816	total: 2m 32s	remaining: 10m 5s
250:	learn: 1.5760575	total: 3m 13s	remaining: 9m 37s
300:	learn: 1.5699575	total: 3m 54s	remaining: 9m 3s
350:	learn: 1.5652123	total: 4m 34s	remaining: 8m 28s
400:	learn: 1.5610414	total: 5m 15s	remaining: 7m 51s
450:	learn: 1.5571667	total: 5m 55s	remaining: 7m 13s
500:	learn: 1.5536433	total: 6m 33s	remaining: 6m 32s
550:	learn: 1.5501641	total: 7m 9s	remaining: 5m 49s
600:	learn: 1.5472859	total: 7m 44s	remaining: 5m 8s
650:	

In [ ]:
# ==================================================
# FINAL MODEL COMPARISON RESULTS
# ==================================================
#               Macro F1 Weighted F1 Macro PR-AUC Macro ROC AUC Top-3 Accuracy Log-Loss
# Random Forest   0.3481      0.4659       0.3857        0.8307         0.8139   1.7902
# LightGBM        0.3552      0.3908       0.3879        0.8397         0.7437   1.6546
# XGBoost         0.3580      0.3943       0.3922        0.8415         0.7460   1.6438
# CatBoost        0.3621      0.3978       0.3968        0.8442         0.7498   1.6333

Genre model test

In [ ]:
# check current directory
import os
print(os.getcwd())

c:\Users\627700\Documents\Music-Analytics-Prediction-RAG\notebook


In [ ]:
# load in the catboost model
import pickle
with open(
        "models/catboost_genre_model_features_selected.pkl",
        "rb"
    ) as file:
        genre_model =  pickle.load(file)

In [ ]:
# double check prediction across all classes

predictions = genre_model.predict(X_test).flatten().astype(int)
actual = np.asarray(y_test).flatten().astype(int)

print("Predicted classes:")
print(pd.Series(predictions).value_counts().sort_index())

print("\nActual classes:")
print(pd.Series(actual).value_counts().sort_index())

Predicted classes:
0     81169
1     65723
2    121279
3     48260
4     85947
5     65890
6     86822
7     47284
8     63456
9     49267
Name: count, dtype: int64

Actual classes:
0     59921
1     17493
2    192275
3     20859
4     45461
5     61214
6     53248
7     69212
8     23988
9    171426
Name: count, dtype: int64


Creating a manual row and checking data pipeline processes correctly

In [30]:
# take raw test row
test_row = pd.DataFrame([{
    "mfcc_zero_mean": -1100.0,
    "onset_rate": 0,
    "average_loudness": 0,
    "danceability": 0,
    "voice_instrumental": 0,
    "mood_acoustic": 0,
    "mood_electronic": 0,
    "tuning_equal_tempered_deviation": 0,
    "bpm": 200,
    "tuning_frequency": 440.0,
    "timbre": 1,
    "dynamic_complexity": 40.0,
    "mood_aggressive_prob": 0,
    "mood_happy_prob": 0
}])

In [18]:
TRAINING_COLUMNS_ORDER = [
    "bpm",
    "danceability",
    "onset_rate",
    "average_loudness",
    "dynamic_complexity",
    "mfcc_zero_mean",
    "tuning_frequency",
    "tuning_equal_tempered_deviation",
    "mood_happy_prob",
    "mood_aggressive_prob",
    "mood_acoustic",
    "mood_electronic",
    "timbre",
    "voice_instrumental",
]

In [19]:
def preprocess_features(
    df,
    minmax_scaler,
    standard_scaler
):

    #df = clamp_features(df)


    # MinMax scaling
    minmax_cols = [
        "mood_happy_prob",
        "mood_aggressive_prob",
    ]

    df[minmax_cols] = minmax_scaler.transform(
        df[minmax_cols]
    )


    # Log transformation
    log_cols = [
        "onset_rate",
        "dynamic_complexity",
        "tuning_equal_tempered_deviation",
    ]

    df[log_cols] = np.log1p(
        df[log_cols]
    )


    # Standard scaling
    standard_cols = [
        "bpm",
        "danceability",
        "mfcc_zero_mean",
        "tuning_frequency",
        "onset_rate",
        "dynamic_complexity",
        "tuning_equal_tempered_deviation",
    ]

    df[standard_cols] = standard_scaler.transform(
        df[standard_cols]
    )


    return df[
        TRAINING_COLUMNS_ORDER
    ]

In [35]:
GENRE_LABELS = [
    "Classical",
    "Country",
    "Electronic",
    "Folk",
    "Hip-Hop/R&B",
    "Jazz/Blues",
    "Metal",
    "Pop",
    "Punk",
    "Rock",
]

In [38]:
# take raw test row
test_row = pd.DataFrame([{
    "mfcc_zero_mean": -900,
        "onset_rate": 2,
        "average_loudness": 0.2,
        "danceability": 0.2,
        "voice_instrumental": 0,
        "mood_acoustic": 1,
        "mood_electronic": 0,
        "tuning_equal_tempered_deviation": 0.05,
        "bpm": 70,
        "tuning_frequency": 440,
        "timbre": 0,
        "dynamic_complexity": 10,
        "mood_aggressive_prob": 0,
        "mood_happy_prob": 0.2
}])

import joblib
processed_row = preprocess_features(
    test_row,
    joblib.load(
        "models/genre_minmax_scaler.pkl"
    ),
    joblib.load(
        "models/genre_standard_scaler.pkl"
    )
)

print("Processed row:")
print(processed_row)

probabilities = genre_model.predict_proba(processed_row)[0]

print("\nProbabilities:")

for i, probability in enumerate(probabilities):
    print(
        f"{i} - {GENRE_LABELS[i]:15s}: {probability:.6f}"
    )

prediction = int(np.argmax(probabilities))

print("\nPrediction:")
print(prediction, GENRE_LABELS[prediction])

Processed row:
       bpm  danceability  onset_rate  average_loudness  dynamic_complexity  \
0 -2.19335     -4.197688    -0.97683               0.2            2.117028   

   mfcc_zero_mean  tuning_frequency  tuning_equal_tempered_deviation  \
0       -2.691235          0.305608                        -0.842103   

   mood_happy_prob  mood_aggressive_prob  mood_acoustic  mood_electronic  \
0             -0.6                  -1.0              1                0   

   timbre  voice_instrumental  
0       0                   0  

Probabilities:
0 - Classical      : 0.082679
1 - Country        : 0.018001
2 - Electronic     : 0.163445
3 - Folk           : 0.350826
4 - Hip-Hop/R&B    : 0.030906
5 - Jazz/Blues     : 0.139768
6 - Metal          : 0.057795
7 - Pop            : 0.022432
8 - Punk           : 0.097585
9 - Rock           : 0.036563

Prediction:
3 Folk


Compare two different test rows

In [37]:
test_rows = {
    "classical_like": {
        "mfcc_zero_mean": -900,
        "onset_rate": 2,
        "average_loudness": 0.2,
        "danceability": 0.2,
        "voice_instrumental": 0,
        "mood_acoustic": 1,
        "mood_electronic": 0,
        "tuning_equal_tempered_deviation": 0.05,
        "bpm": 70,
        "tuning_frequency": 440,
        "timbre": 0,
        "dynamic_complexity": 10,
        "mood_aggressive_prob": 0,
        "mood_happy_prob": 0.2
    },

    "electronic_like": {
        "mfcc_zero_mean": -500,
        "onset_rate": 20,
        "average_loudness": 0.9,
        "danceability": 2.5,
        "voice_instrumental": 0,
        "mood_acoustic": 0,
        "mood_electronic": 1,
        "tuning_equal_tempered_deviation": 0.2,
        "bpm": 160,
        "tuning_frequency": 440,
        "timbre": 1,
        "dynamic_complexity": 70,
        "mood_aggressive_prob": 0.8,
        "mood_happy_prob": 0.8
    }
}

for name, values in test_rows.items():

    row = pd.DataFrame([values])

    processed = preprocess_features(
        row,
        joblib.load("models/genre_minmax_scaler.pkl"),
        joblib.load("models/genre_standard_scaler.pkl")
    )

    probabilities = genre_model.predict_proba(processed)[0]

    prediction = int(np.argmax(probabilities))

    print("\n" + "=" * 50)
    print(name)
    print("Prediction:", GENRE_LABELS[prediction])

    for i, p in enumerate(probabilities):
        print(f"{GENRE_LABELS[i]:15s}: {p:.6f}")


classical_like
Prediction: Folk
Classical      : 0.082679
Country        : 0.018001
Electronic     : 0.163445
Folk           : 0.350826
Hip-Hop/R&B    : 0.030906
Jazz/Blues     : 0.139768
Metal          : 0.057795
Pop            : 0.022432
Punk           : 0.097585
Rock           : 0.036563

electronic_like
Prediction: Electronic
Classical      : 0.001535
Country        : 0.000010
Electronic     : 0.932170
Folk           : 0.000588
Hip-Hop/R&B    : 0.000545
Jazz/Blues     : 0.000202
Metal          : 0.008104
Pop            : 0.006074
Punk           : 0.044494
Rock           : 0.006278


Confusion matrix for random forest

In [13]:
from sklearn.metrics import confusion_matrix

y_t = y_test
y_p = rf_model.predict(X_test)

matrix = confusion_matrix(y_t, y_p.ravel())

# Define class labels in order
class_labels = ['Classical', 'Country', 'Electronic', 'Folk', 'Hip-Hop/R&B', 'Jazz/Blues', 'Metal', 'Pop', 'Punk', 'Rock'] 

# Electronic     961377 = 2
# Rock           857129 = 9
# Pop            346058 = 7
# Jazz/Blues     306069 = 5
# Classical      299607 = 0
# Metal          266239 = 6
# Hip-Hop/R&B    227304 = 4
# Punk           119940 = 8
# Folk           104295 = 3
# Country         87465 = 1

matrix_df = pd.DataFrame(
    matrix, 
    index=[f"Actual {label}" for label in class_labels], 
    columns=[f"Predicted {label}" for label in class_labels]
)

matrix_df

,Predicted Classical,Predicted Country,Predicted Electronic,Predicted Folk,Predicted Hip-Hop/R&B,Predicted Jazz/Blues,Predicted Metal,Predicted Pop,Predicted Punk,Predicted Rock
Actual Classical,40552,51,9152,204,88,3718,317,374,1,5464
Actual Country,659,1277,1159,266,131,1884,20,1259,2,10836
Actual Electronic,6629,157,140610,219,5868,5491,4225,2465,106,26505
Actual Folk,1892,407,3498,723,188,2909,138,1150,5,9949
Actual Hip-Hop/R&B,412,78,21550,55,14495,1114,211,1113,13,6420
Actual Jazz/Blues,6908,418,12914,406,1001,22753,186,1436,9,15183
Actual Metal,557,14,7901,13,102,317,23885,145,385,19929
Actual Pop,2202,528,19855,371,2284,3419,758,5862,58,33875
Actual Punk,166,35,3827,31,212,304,3654,175,777,14807
Actual Rock,4824,763,30535,597,1766,6858,11358,4651,699,109375


Save confusion matrix

In [14]:
matrix_df.to_csv('confusion_matrix/genre_features_selected_rf_confusion_matrix.csv', index=True)

Confusion Matrix for Catboost

In [15]:
from sklearn.metrics import confusion_matrix

y_t = y_test
y_p = catboost_model.predict(X_test)

matrix = confusion_matrix(y_t, y_p.ravel())

# Define class labels in order
class_labels = ['Classical', 'Country', 'Electronic', 'Folk', 'Hip-Hop/R&B', 'Jazz/Blues', 'Metal', 'Pop', 'Punk', 'Rock'] 

# Electronic     961377 = 2
# Rock           857129 = 9
# Pop            346058 = 7
# Jazz/Blues     306069 = 5
# Classical      299607 = 0
# Metal          266239 = 6
# Hip-Hop/R&B    227304 = 4
# Punk           119940 = 8
# Folk           104295 = 3
# Country         87465 = 1

matrix_df = pd.DataFrame(
    matrix, 
    index=[f"Actual {label}" for label in class_labels], 
    columns=[f"Predicted {label}" for label in class_labels]
)

matrix_df

,Predicted Classical,Predicted Country,Predicted Electronic,Predicted Folk,Predicted Hip-Hop/R&B,Predicted Jazz/Blues,Predicted Metal,Predicted Pop,Predicted Punk,Predicted Rock
Actual Classical,44240,1131,2974,3734,565,4298,1219,584,326,850
Actual Country,648,9572,259,2423,459,1538,156,1075,513,850
Actual Electronic,13674,4046,85967,6916,32389,11433,13958,11835,6420,5637
Actual Folk,1991,5213,1013,5917,882,2578,505,1095,690,975
Actual Hip-Hop/R&B,649,1781,5598,1543,28354,1972,770,2566,1374,854
Actual Jazz/Blues,7947,6363,3776,6167,3951,26237,813,2332,1234,2394
Actual Metal,1253,439,2827,717,713,614,34652,867,8180,2986
Actual Pop,2837,13034,7020,6778,8387,4838,2996,12779,4824,5719
Actual Punk,328,1087,1400,658,1257,611,4912,837,10655,2243
Actual Rock,7602,23057,10445,13407,8990,11771,26841,13314,29240,26759


Save the confusion matrix

In [16]:
matrix_df.to_csv('confusion_matrix/genre_features_selected_catboost_confusion_matrix.csv', index=True)

Calculate feature importance for random forest

In [17]:
importances = rf_model.feature_importances_

# Map to feature names and sort
fi_df = pd.DataFrame({
    'feature_names': X_train.columns,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(fi_df)

                      feature_names  importance
5                    mfcc_zero_mean    0.116988
2                        onset_rate    0.109614
1                      danceability    0.104411
3                  average_loudness    0.102916
4                dynamic_complexity    0.092875
0                               bpm    0.091679
7   tuning_equal_tempered_deviation    0.091659
9              mood_aggressive_prob    0.081877
8                   mood_happy_prob    0.081651
6                  tuning_frequency    0.056893
13               voice_instrumental    0.019208
10                    mood_acoustic    0.018751
11                  mood_electronic    0.016813
12                           timbre    0.014665


In [ ]:
# feature_names  importance
# 5                    mfcc_zero_mean    0.116988
# 2                        onset_rate    0.109614
# 1                      danceability    0.104411
# 3                  average_loudness    0.102916
# 4                dynamic_complexity    0.092875
# 0                               bpm    0.091679
# 7   tuning_equal_tempered_deviation    0.091659
# 9              mood_aggressive_prob    0.081877
# 8                   mood_happy_prob    0.081651
# 6                  tuning_frequency    0.056893
# 13               voice_instrumental    0.019208
# 10                    mood_acoustic    0.018751
# 11                  mood_electronic    0.016813
# 12                           timbre    0.014665

Calculate feature importance for catboost model

In [18]:
importances = catboost_model.feature_importances_

# Map to feature names and sort
fi_df = pd.DataFrame({
    'feature_names': X_train.columns,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(fi_df)

                      feature_names  importance
5                    mfcc_zero_mean   18.605185
2                        onset_rate   14.126526
3                  average_loudness    8.496914
1                      danceability    7.702493
13               voice_instrumental    7.407503
7   tuning_equal_tempered_deviation    6.470061
11                  mood_electronic    6.261348
12                           timbre    5.457738
10                    mood_acoustic    5.220993
0                               bpm    5.159143
6                  tuning_frequency    4.623779
4                dynamic_complexity    4.542584
9              mood_aggressive_prob    4.057507
8                   mood_happy_prob    1.868228


In [ ]:
# feature_names  importance
# 5                    mfcc_zero_mean   18.605185
# 2                        onset_rate   14.126526
# 3                  average_loudness    8.496914
# 1                      danceability    7.702493
# 13               voice_instrumental    7.407503
# 7   tuning_equal_tempered_deviation    6.470061
# 11                  mood_electronic    6.261348
# 12                           timbre    5.457738
# 10                    mood_acoustic    5.220993
# 0                               bpm    5.159143
# 6                  tuning_frequency    4.623779
# 4                dynamic_complexity    4.542584
# 9              mood_aggressive_prob    4.057507
# 8                   mood_happy_prob    1.868228